In [ ]:
import os
import librosa as lb
import tensorflow as tf
import tensorflow_hub as tf_hub 
import tensorflow.keras as keras # type: ignore #
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import pandas as pd
import numpy as np


In [ ]:
class WaveformSegmenter:
    def __init__(self):
        pass

    def segmentize(self, file_path, sr, segment_seconds, overlap_seconds, is_mono = False):
        segments = []

        try:
            waveform, sample_rate = lb.load(file_path, sr=sr, mono=is_mono)

            segment_samples = segment_seconds * sample_rate
            overlap_samples = segment_samples - (overlap_seconds * sample_rate)

            for start in range(0, len(waveform), overlap_samples):
                segment = waveform[start: start + segment_samples]
                segments.append(segment)

            return segments

        except:
            print(f"Error has occured at path {file_path}")
            return None
    
class Yamnet:
    def __init__(self):
        self.__model = tf_hub.load('https://tfhub.dev/google/yamnet/1')

    def get_mean_pooled_waveform_embeddings(self, waveform, axis=0):
        _, embeddings, _ = self.__model(waveform)
        mean_pooled_embeddings = tf.reduce_mean(embeddings, axis=axis)
        
        return mean_pooled_embeddings.numpy()

waveform_segmenter = WaveformSegmenter()
yamnet = Yamnet()

In [ ]:
genre_map = {
    "blues": 0,
    "classical": 1,
    "country": 2,
    "disco": 3,
    "hiphop": 4,
    "jazz": 5,
    "metal": 6,
    "pop": 7,
    "reggae": 8,
    "rock": 9
}

X = []
y = []
dataset_folder_name = "genres_original"

for genre_folder_name in os.listdir(dataset_folder_name):
    genre_path = os.path.join(dataset_folder_name, genre_folder_name)
    for wav_file in tqdm(os.listdir(genre_path), desc=f"Getting the audio paths in folder {genre_folder_name}"):
        try:
            wav_file_path = os.path.join(genre_path, wav_file)
            X.append(wav_file_path)
            y.append(genre_map[genre_folder_name])
        except:
            print(f"Error has occured at path {wav_file_path}")

Getting the audio paths in folder rock: 100%|██████████| 100/100 [00:00<?, ?it/s]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    random_state=42,
    test_size=0.25,
    stratify=y
)

In [5]:
segmented_X_train = []
segmented_y_train = []
segmented_X_test = []
segmented_y_test = []

for wav_file_path, label in zip(tqdm(X_train, desc="Segmentizing X_train and y_train"), y_train):
    segments = waveform_segmenter.segmentize(
        file_path=wav_file_path, 
        sr=16000, 
        segment_seconds=5, 
        overlap_seconds=3, 
        is_mono=True
    )
    if segments != None:
        segmented_X_train.extend(segments)
        segmented_y_train.extend([label] * len(segments))

for wav_file_path, label in zip(tqdm(X_test, desc="Segmentizing X_test and y_test"), y_test):
    segments = waveform_segmenter.segmentize(
        file_path=wav_file_path, 
        sr=16000, 
        segment_seconds=5, 
        overlap_seconds=3, 
        is_mono=True
    )
    if segments != None:
        segmented_X_test.extend(segments)
        segmented_y_test.extend([label] * len(segments))

embbeded_segments_X_train = []
embbeded_segments_X_test = []

for segment in tqdm(segmented_X_train, desc="Getting the embeddings in segmented_X_train"):
    embeddings = yamnet.get_mean_pooled_waveform_embeddings(segment)
    embbeded_segments_X_train.append(embeddings)

for segment in tqdm(segmented_X_test, desc="Getting the embeddings in segmented_X_test"):
    embeddings = yamnet.get_mean_pooled_waveform_embeddings(segment)
    embbeded_segments_X_test.append(embeddings)
    

Segmentizing X_train and y_train:   0%|          | 0/750 [00:00<?, ?it/s]c:\Users\Admin\10-class-genre-classification-model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Segmentizing X_train and y_train:  21%|██        | 155/750 [00:09<00:19, 30.90it/s]C:\Users\Admin\AppData\Local\Temp\ipykernel_2864\2543725484.py:9: UserWarning: PySoundFile failed. Trying audioread instead.
  waveform, sample_rate = lb.load(file_path, sr=sr, mono=is_mono)
c:\Users\Admin\10-class-genre-classification-model\venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Segmentizing X_train and y_train:  22%|██▏       | 163/750 [00:09<00:17

Error has occured at path genres_original\jazz\jazz.00054.wav


Getting the embeddings in segmented_X_test: 100%|██████████| 4000/4000 [01:44<00:00, 38.34it/s]


In [6]:
embbeded_segments_X_train = np.array(embbeded_segments_X_train)
embbeded_segments_X_test = np.array(embbeded_segments_X_test)
segmented_y_train = np.array(segmented_y_train)
segmented_y_test = np.array(segmented_y_test)

model = keras.Sequential([
    keras.layers.Input(shape=(1024,)),
    keras.layers.Dense(256, activation=keras.activations.relu),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation=keras.activations.relu),
    keras.layers.Dense(len(genre_map), activation=keras.activations.softmax)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.sparse_categorical_accuracy
    ]
)

history = model.fit(
    embbeded_segments_X_train, 
    segmented_y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=10,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ]
)

Epoch 1/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 1.0000 - sparse_categorical_accuracy: 0.6848 - val_loss: 0.8517 - val_sparse_categorical_accuracy: 0.7336
Epoch 2/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.6758 - sparse_categorical_accuracy: 0.7807 - val_loss: 0.8671 - val_sparse_categorical_accuracy: 0.7332
Epoch 3/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.5777 - sparse_categorical_accuracy: 0.8083 - val_loss: 0.8357 - val_sparse_categorical_accuracy: 0.7332
Epoch 4/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.5202 - sparse_categorical_accuracy: 0.8288 - val_loss: 0.8866 - val_sparse_categorical_accuracy: 0.7344
Epoch 5/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4881 - sparse_categorical_accuracy: 0.8347 - val_loss: 0.8758 - val_sparse_categorical_accuracy: 0.7353
Epoch 6/100
958/958 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.4543 - sparse_categorical_accuracy: 0.8474 - val_loss: 0.8661 - val_sparse_categorical_accuracy: 0.747

In [7]:
print("Classification report of the model, such as F1, Precision and Recall, and \nConfusion Matrix to check, on what classes the model most confuses with\n")

genre_labels = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

y_preds = model.predict(embbeded_segments_X_test)
y_preds_indexes = np.argmax(y_preds, axis=1)

print(classification_report(
    segmented_y_test,
    y_preds_indexes,
    target_names=genre_map
))

conf_matrix = confusion_matrix(segmented_y_test, y_preds_indexes)

confusion_matrix_dataframe = pd.DataFrame(
    conf_matrix,
    index=genre_labels,
    columns=genre_labels
)

print(confusion_matrix_dataframe)

Classification report of the model, such as F1, Precision and Recall, and 
Confusion Matrix to check, on what classes the model most confuses with

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
              precision    recall  f1-score   support

       blues       0.74      0.72      0.73       400
   classical       0.91      0.96      0.93       400
     country       0.73      0.74      0.73       400
       disco       0.77      0.62      0.69       400
      hiphop       0.78      0.79      0.79       400
        jazz       0.83      0.89      0.86       400
       metal       0.82      0.84      0.83       400
         pop       0.63      0.78      0.70       400
      reggae       0.75      0.73      0.74       400
        rock       0.55      0.45      0.49       400

    accuracy                           0.75      4000
   macro avg       0.75      0.75      0.75      4000
weighted avg       0.75      0.75      0.75      4000

           blues  classical  country  disco  hiphop 

In [ ]:
print("\nTesting with segmentation of the audio to be predicted then turn the segments into embeddings")

z = []

waveform_segments = waveform_segmenter.segmentize(
                file_path="test_samples/Justin Bieber - Peaches ft. Daniel Caesar, Giveon.mp3",
                segment_seconds= 5,
                overlap_seconds=3, 
                sr=16000, 
                is_mono=True
)

for waveform_segment in waveform_segments:
    waveform_segment_embbedings = yamnet.get_mean_pooled_waveform_embeddings(waveform=waveform_segment)
    z.append(waveform_segment_embbedings)

z = np.array(z)
y_pred = model.predict(z)
mean_pooled_y_pred = np.mean(y_pred, axis=0)
class_predicted = genre_labels[np.argmax(mean_pooled_y_pred)]

print(class_predicted)


Testing with segmentation of the audio to be predicted then turn the segments into embeddings
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
pop


In [ ]:
print("\nTesting with embeddings only.")


waveform, sr = lb.load("test_samples/Justin Bieber - Peaches ft. Daniel Caesar, Giveon.mp3", sr=16000, mono=True)
waveform_embeddings = yamnet.get_mean_pooled_waveform_embeddings(waveform=waveform)

waveform_embeddings = tf.expand_dims(waveform_embeddings, axis=0)
preds = model.predict(waveform_embeddings)
class_pred = genre_labels[np.argmax(preds)]

print(class_pred)



Testing with embeddings only.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
pop


In [14]:
print("Saving the models into tflite format:")

if not os.path.exists("genre_classification_model.tflite"):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    to_tf_lite = converter.convert()

    with open("genre_classification_model.tflite", "wb") as f:
        f.write(to_tf_lite)

    print("Done — genre_classification_model.tflite saved")


Saving the models into tflite format:
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp0a02wvsy\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp0a02wvsy\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmp0a02wvsy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1972186096528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972186096144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300158096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300157904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300157712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972186096912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972186096336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300160976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300158480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1972300159824: TensorSpec(shape=(), dtype=tf.resource, name=None)
Done — genre_

In [15]:
if not os.path.exists("yamnet_model.tflite"):
    yamnet_model = tf_hub.load("https://tfhub.dev/google/yamnet/1")

    @tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
    def yamnet_fn(waveform):
        scores, embeddings, spectrogram = yamnet_model(waveform)
        mean_pooled_embeddings = tf.reduce_mean(embeddings, axis=0)
        return {"scores": scores, "embeddings": mean_pooled_embeddings, "spectrogram": spectrogram}

    tf.saved_model.save(
        yamnet_model,
        "yamnet_saved_model",
        signatures={"serving_default": yamnet_fn}
    )

    converter = tf.lite.TFLiteConverter.from_saved_model(
        "yamnet_saved_model",
        signature_keys=["serving_default"]
    )

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]

    tflite_model = converter.convert()

    with open("yamnet_model.tflite", "wb") as f:
        f.write(tflite_model)

    print("Done — yamnet_model.tflite saved")

INFO:tensorflow:Assets written to: yamnet_saved_model\assets


INFO:tensorflow:Assets written to: yamnet_saved_model\assets


Done — yamnet_model.tflite saved
